Complete Kaggle-ready pipeline for "House Prices - Advanced Regression Techniques" (Ames dataset).
This single-file script performs:
- Data loading
- Simple EDA notes (print statements)
- Feature engineering & preprocessing
- Out-of-fold training with multiple models (Lasso, RandomForest, XGBoost, LightGBM, CatBoost)
- Blending / simple stacking
- Generate submission.csv


Notes:
- Run on Kaggle kernel or locally (ensure packages installed)
- If running locally, install: pip install numpy pandas scikit-learn xgboost lightgbm catboost
- This is a strong baseline that uses ensembling; further tuning/feature engineering will improve leaderboard score.


Author: Hammad Farooq


In [ ]:
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# 1) Load data
# ============================================================
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Kaggle dataset paths (House Prices)
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

In [ ]:
# ============================================================
# 2) Preprocessing
# ============================================================

# Target log-transform
y = np.log1p(train['SalePrice'].values)

# Drop target + Id
X = train.drop(['Id','SalePrice'], axis=1)
X_test = test.drop(['Id'], axis=1)

# Fill missing values
X = X.fillna("None")
X_test = X_test.fillna("None")

# Encode categoricals with LabelEncoder
for col in X.select_dtypes(include=['object']).columns:
    lbl = LabelEncoder()
    lbl.fit(list(X[col].astype(str)) + list(X_test[col].astype(str)))
    X[col] = lbl.transform(list(X[col].astype(str)))
    X_test[col] = lbl.transform(list(X_test[col].astype(str)))

# Align columns
X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

# Convert to numpy
X = X.values
X_test = X_test.values

print("Shapes -> X:", X.shape, ", y:", y.shape, ", X_test:", X_test.shape)

In [ ]:
# ============================================================
# 3) Define models
# ============================================================
models = {
    "Lasso": Lasso(alpha=0.0005, random_state=42),
    "Ridge": Ridge(alpha=10, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=0),
    "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_state=42, verbose=0)
}

# ============================================================
# 4) Helper function for OOF predictions
# ============================================================
def get_oof_predictions(model, X, y, X_test, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_train = np.zeros((X.shape[0],1))
    oof_test_skf = np.empty((n_splits, X_test.shape[0]))
    
    for i, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        model.fit(X_train, y_train)
        oof_train[val_idx] = model.predict(X_val).reshape(-1,1)
        oof_test_skf[i,:] = model.predict(X_test)
    
    oof_test = oof_test_skf.mean(axis=0).reshape(-1,1)
    return oof_train, oof_test

# ============================================================
# 5) Training base models (OOF)
# ============================================================
oof_train_list = []
oof_test_list = []
cv_scores = {}

for name, model in models.items():
    try:
        print('\nTraining', name)
        oof_train, oof_test = get_oof_predictions(model, X, y, X_test)
        score = np.sqrt(mean_squared_error(y, oof_train.ravel()))
        cv_scores[name] = score
        print(f"OOF RMSE for {name}: {score:.5f}")
        oof_train_list.append(oof_train)
        oof_test_list.append(oof_test)
    except Exception as e:
        print('Skipping', name, 'because', e)

print('\nCV scores (on log target):', cv_scores)

In [ ]:
# ============================================================
# 6) Stacking with Lasso as blender
# ============================================================
if len(oof_train_list) > 0:
    X_oof = np.hstack(oof_train_list)
    X_test_oof = np.hstack(oof_test_list)
    print('Stacked OOF shape:', X_oof.shape)

    blender = Lasso(alpha=0.0005, random_state=42)
    blender.fit(X_oof, y)
    preds_oof = blender.predict(X_test_oof)
else:
    print("⚠️ No OOF features created, fallback to zeros")
    preds_oof = np.zeros(X_test.shape[0])

# ============================================================
# 7) Simple average ensemble of all models
# ============================================================
full_preds = []
for name, model in models.items():
    try:
        print('Fitting full', name)
        model.fit(X, y)
        p = model.predict(X_test)
        full_preds.append(p)
    except Exception as e:
        print('Skipping full fit for', name, e)

if len(full_preds) > 0:
    avg_full = np.mean(np.vstack(full_preds), axis=0)
    final_preds_log = 0.5 * preds_oof.ravel() + 0.5 * avg_full.ravel()
else:
    final_preds_log = preds_oof.ravel()

# ============================================================
# 8) Convert back from log
# ============================================================
final_preds = np.expm1(final_preds_log)

print("Final predictions ready! Shape:", final_preds.shape)

In [ ]:
# ============================================================
# 9) Save submission
# ============================================================
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_preds
})
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv saved!")